In [1]:

# ============================================================
# 1. IMPORTS
# ============================================================

from langchain.agents import create_agent
from langchain.tools import tool

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama, OllamaEmbeddings


# ============================================================
# 2. LOAD DOCUMENTS
# ============================================================

loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")
documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")
documents_diet = loader_diet.load()


# ============================================================
# 3. SPLIT DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)


# ============================================================
# 4. CREATE EMBEDDINGS AND FAISS VECTOR STORES
# ============================================================

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore_fruits = FAISS.from_documents(
    chunks_fruits,
    embedding_model
)

vectorstore_diet = FAISS.from_documents(
    chunks_diet,
    embedding_model
)


# ============================================================
# 5. CREATE RETRIEVERS
# ============================================================

fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 2}
)

nutrition_retriever = vectorstore_diet.as_retriever(
    search_kwargs={"k": 2}
)


# ============================================================
# 6. INITIALIZE CHAT MODEL
# ============================================================

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)


# ============================================================
# 7. CREATE RETRIEVAL TOOLS
# ============================================================

@tool
def search_fruits(query: str) -> str:
    """Search fruit and vegetable documents for information
    about fruits, vegetables, colours, seeds, taste and cooking."""

    docs = fruits_retriever.invoke(query)

    return "\n\n".join(
        doc.page_content for doc in docs
    )


@tool
def search_nutrition(query: str) -> str:
    """Search nutrition documents for information about
    vitamins, minerals, proteins, carbohydrates, fats,
    fibre, hydration and dietary requirements."""

    docs = nutrition_retriever.invoke(query)

    return "\n\n".join(
        doc.page_content for doc in docs
    )


# ============================================================
# 8. CREATE AGENT
# ============================================================

agent = create_agent(
    model=llm,
    tools=[search_fruits, search_nutrition],
    system_prompt="""
    You are a helpful knowledge assistant.

    You have access to two retrieval tools:
    1. search_fruits
    2. search_nutrition

    Decide which tool is appropriate for the user's question.
    You may use both tools if the question requires information
    from both sources.

    Use the retrieved information to answer the user.
    Do not invent information that is not present in the documents.
    """
)


# ============================================================
# 9. ASK QUESTION
# ============================================================

question = "What nutrients do apples provide?"


# ============================================================
# 10. INVOKE AGENT
# ============================================================

result = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})


# ============================================================
# 11. PRINT FINAL ANSWER
# ============================================================

print("Final Answer:")
print(result["messages"][-1].content)


C:\Users\Yoga\AppData\Local\Temp\ipykernel_11108\4064431952.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Final Answer:
Apples provide several important nutrients. Here are some key nutrients found in apples:

1. **Vitamins**: Apples are a good source of vitamin C, which is essential for immune function, collagen production, and the absorption of iron. They also contain small amounts of vitamin A, which is important for vision, immune function, and skin health.

2. **Minerals**: Apples are a good source of several minerals, including:
   - **Potassium**: Potassium is crucial for maintaining fluid balance in the body, helping to regulate blood pressure, and supporting nerve and muscle function.
   - **Fiber**: Apples contain soluble and insoluble fiber, which helps in digestion, reduces cholesterol levels, and aids in weight management.
   - **Manganese**: Manganese is a trace mineral that plays a role in bone health, immune function, and the regulation of blood sugar levels.
   - **Phenolic Compounds**: These compounds, found in apples, have antioxidant properties and may help reduce infla